In [3]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


## Importing Library


In [ ]:
import os
import cv2
import numpy as np
import pickle

In [2]:
image_folder = "/content/drive/MyDrive/OD_Project/food_dataset/food_dataset/images"

images = os.listdir(image_folder)

print(len(images))
print(images[:5])

4747
['Sante_9c10bca1.jpg', 'ROYAL_STEP_USB_Portable_Electric_USB_Juice_Maker_Juicer_Bott_b6583606.jpg', 'Sama_raw_milk_chocolate_for_cooking_200g_cd6af157.jpg', 'Sacrix_ea5f1a02.jpg', 'San_Benedetto_Elite_Still_Water_-_500ML_c5708e71.jpg']


### Clean image

In [ ]:
import cv2
import os

bad_images = []

for img_name in images:
    img_path = os.path.join(image_folder, img_name)

    img = cv2.imread(img_path)

    if img is None:
        bad_images.append(img_name)

print("image not clean: ", len(bad_images))

image not clean:  0


### crop image

In [ ]:
import os

crop_folder = "/content/drive/MyDrive/OD_Project/food_dataset/cropped_images"
os.makedirs(crop_folder, exist_ok=True)

In [ ]:
for img_name in images:
    img_path = os.path.join(image_folder, img_name)

    img = cv2.imread(img_path)

    if img is None:
        continue

    # === Crop ===
    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    _, thresh = cv2.threshold(gray, 10, 255, cv2.THRESH_BINARY)
    coords = cv2.findNonZero(thresh)

    if coords is not None:
        x, y, w, h = cv2.boundingRect(coords)
        img = img[y:y+h, x:x+w]

    # === Save ===
    save_path = os.path.join(crop_folder, img_name)
    cv2.imwrite(save_path, img)

print("done crop")

done crop


### resize image

In [ ]:
cropped_folder = "/content/drive/MyDrive/OD_Project/food_dataset/cropped_images"
resized_folder = "/content/drive/MyDrive/OD_Project/food_dataset/resized_images"

os.makedirs(resized_folder, exist_ok=True)

In [ ]:
images = os.listdir(cropped_folder)

for img_name in images:
    img_path = os.path.join(cropped_folder, img_name)

    img = cv2.imread(img_path)

    if img is None:
        continue

    # === Resize + Padding ===
    h, w, _ = img.shape
    scale = 224 / max(h, w)

    new_h = int(h * scale)
    new_w = int(w * scale)

    img_resized = cv2.resize(img, (new_w, new_h))

    new_img = np.zeros((224, 224, 3), dtype=np.uint8)

    y_offset = (224 - new_h) // 2
    x_offset = (224 - new_w) // 2

    new_img[y_offset:y_offset+new_h, x_offset:x_offset+new_w] = img_resized

    # === Save ===
    save_path = os.path.join(resized_folder, img_name)
    cv2.imwrite(save_path, new_img)

print("finish resize")

finish resize


In [ ]:

from tensorflow.keras.applications import ResNet50
from tensorflow.keras.models import Model
from tensorflow.keras.applications.resnet50 import preprocess_input

# ====== Paths ======
resized_folder = "/content/drive/MyDrive/OD_Project/food_dataset/resized_images"
output_file = "/content/drive/MyDrive/OD_Project/food_dataset/image_features.pkl"

# ====== Load Model ======
base_model = ResNet50(weights='imagenet')
model = Model(inputs=base_model.input, outputs=base_model.layers[-2].output)

# ====== Read Images ======
images = os.listdir(resized_folder)

# ====== Feature Storage ======
image_features = {}

# ====== Loop on Images ======
for img_name in images:
    img_path = os.path.join(resized_folder, img_name)

    img = cv2.imread(img_path)

    if img is None:
        continue

    # Convert to RGB
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

    # Preprocess for ResNet
    img = preprocess_input(img)

    # Expand dims
    img = np.expand_dims(img, axis=0)

    # Extract Features
    features = model.predict(img, verbose=0)

    # Save feature (remove batch dim)
    image_features[img_name] = features[0]

# ====== Save to File ======
with open(output_file, "wb") as f:
    pickle.dump(image_features, f)

print("done feature excration")

done feature excration


### read from PKI

In [3]:
file_path = "/content/drive/MyDrive/OD_Project/food_dataset/image_features.pkl"

with open(file_path, "rb") as f:
    data = pickle.load(f)

print("file is readed")


print("number of photos: ", len(data))

for i, (img_name, features) in enumerate(data.items()):
    print("Name of Photo", img_name)
    print("shape of future: ", features.shape)
    print("first 10 values: ", features[:10])
    print("-" * 50)

    if i == 4:
        break

file is readed
number of photos:  4747
Name of Photo Eva_Optimum_Care_Recipe_Silk_Shine_Blend_Shampoo_Marshmallow_b31f1a53.jpg
shape of future:  (2048,)
first 10 values:  [0.65023667 0.45194802 1.0369515  0.01595325 0.9581611  0.
 0.07307959 0.48118588 0.1187553  0.2524433 ]
--------------------------------------------------
Name of Photo El_Shamadan_Joy_Biscuit_Wafer_Filled_With_Flavored_Vanillia__a9ecae75.jpg
shape of future:  (2048,)
first 10 values:  [0.26067823 0.50586194 0.66988623 0.13277166 0.5339758  0.00684342
 1.4121184  0.30804026 0.3440169  0.46018896]
--------------------------------------------------
Name of Photo Fajita_Spices_Spicy_50g_SPICEKICK_Natural_Balanced_Seasoning_823331c7.jpg
shape of future:  (2048,)
first 10 values:  [0.11355515 0.09538256 0.7051237  0.02384971 0.         0.03357036
 0.46103233 0.6713783  0.         1.1522021 ]
--------------------------------------------------
Name of Photo Extra_Food_Worcester_Sauce_300ml_Teriyaki_and_Pomegranate_Fl_be616d

In [2]:
with open("/content/drive/MyDrive/OD_Project/food_dataset/image_features.pkl", "rb") as f:
    image_features = pickle.load(f)

print("number of photos: ", len(image_features))

number of photos:  4747
